# V6.4 Post-Clean Dataset Gate + Strict-IoU Residual Audit

Notebook này **không sửa nhãn và không train model mới**.

Mục tiêu:

1. Kiểm tra V6.4 hiện tại có sạch và tự nhất quán không.
2. Đo strict localization trên **VAL** bằng `best.pt`.
3. Tách residual thành:
   - detection miss,
   - center localization error,
   - bbox extent/shape error.
4. Xuất contact sheet để kiểm tra bằng mắt trước khi quyết định bước tiếp theo.

Checkpoint mặc định:

`/home/jovyan/projects/yolov11m/datasets/v6p4_clean/v6p4_clean_official/best.pt`

Nguyên tắc: có thể kiểm tra integrity của `train/val/test`, nhưng **không dùng TEST để tune hay chọn hướng**.

In [ ]:
from pathlib import Path
import os, sys, json, math, hashlib, shutil, subprocess
import numpy as np
import pandas as pd

WEIGHTS = Path("/home/jovyan/projects/yolov11m/datasets/v6p4_clean/v6p4_clean_official/best.pt")

# Nếu auto-discovery sai, điền path data.yaml thật.
DATA_YAML_OVERRIDE = None

SEARCH_ROOTS = [
    WEIGHTS.parent,
    WEIGHTS.parent.parent,
    Path("/home/jovyan/projects/yolov11m/datasets"),
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/tmp"),
]

OUT = Path("/home/jovyan/projects/yolov11m/audits/v6p4_post_clean_gate")
OUT.mkdir(parents=True, exist_ok=True)

NAMES_EXPECTED = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

RUN_PHASH_NEAR_DUP = True
PHASH_MAX_HAMMING = 4
DUP_GT_IOU = 0.95

RUN_MODEL_VAL = True
IMGSZ = 1024
VAL_BATCH = 8
DEVICE = 0

RUN_RESIDUAL_ANALYSIS = True
RESIDUAL_CONF = 0.10
RESIDUAL_BATCH = 8
GOOD_IOU = 0.75
CENTER_MATCH_MAX = 0.25
MAX_CONTACT_SHEETS = 180

print("WEIGHTS:", WEIGHTS)
print("OUT:", OUT)
assert WEIGHTS.exists(), f"Không thấy best.pt: {WEIGHTS}"

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.4.172", "pyyaml", "pillow", "pandas", "imagehash", "tqdm"
])

import yaml
import imagehash
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from ultralytics import YOLO

print("Imports OK")

In [ ]:
def norm_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names or [])

def resolve_yaml_paths(yp):
    yp = Path(yp).resolve()
    cfg = yaml.safe_load(yp.read_text())
    base = Path(cfg.get("path", yp.parent))
    if not base.is_absolute():
        base = (yp.parent / base).resolve()

    out = {"yaml": yp, "cfg": cfg, "base": base}
    for split in ["train", "val", "test"]:
        raw = cfg.get(split)
        if raw is None:
            out[split] = None
        else:
            p = Path(raw)
            out[split] = p if p.is_absolute() else (base / p).resolve()
    return out

def score_yaml(yp):
    try:
        z = resolve_yaml_paths(yp)
        names = norm_names(z["cfg"].get("names"))
        score = 0
        if names == NAMES_EXPECTED:
            score += 20
        if "v6p4" in str(yp).lower():
            score += 10
        for s in ["train", "val", "test"]:
            p = z[s]
            if p is not None and p.exists():
                score += 7
        if z["base"].exists():
            score += 2
        return score, z
    except Exception:
        return -1, None

if DATA_YAML_OVERRIDE:
    DATA_YAML = Path(DATA_YAML_OVERRIDE)
    assert DATA_YAML.exists(), DATA_YAML
else:
    seen = set()
    candidates = []

    for root in SEARCH_ROOTS:
        root = Path(root)
        if not root.exists():
            continue

        for p in [root / "data.yaml", root / "dataset.yaml"]:
            if p.exists():
                seen.add(str(p.resolve()))

        try:
            for p in root.glob("**/data.yaml"):
                sp = str(p.resolve())
                if sp in seen:
                    continue
                if "v6p4" in sp.lower() or root == WEIGHTS.parent:
                    seen.add(sp)
        except Exception:
            pass

    for sp in seen:
        p = Path(sp)
        sc, z = score_yaml(p)
        if sc >= 0:
            candidates.append((sc, p, z))

    candidates.sort(key=lambda x: x[0], reverse=True)

    print("Top data.yaml candidates:")
    for sc, p, z in candidates[:15]:
        print(f"  score={sc:>2} | {p}")

    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy data.yaml V6.4. Set DATA_YAML_OVERRIDE ở cell CONFIG."
        )

    DATA_YAML = candidates[0][1]

resolved = resolve_yaml_paths(DATA_YAML)
cfg = resolved["cfg"]
DATA_BASE = resolved["base"]
NAMES = norm_names(cfg.get("names"))

print("\nSELECTED:", DATA_YAML)
print("DATA_BASE:", DATA_BASE)
print("NAMES:", NAMES)
for s in ["train", "val", "test"]:
    print(f"{s:5s}:", resolved[s])

assert NAMES == NAMES_EXPECTED, f"Class taxonomy lệch: {NAMES}"

for s in ["train", "val"]:
    assert resolved[s] is not None and resolved[s].exists(), (
        f"Không resolve được split {s}: {resolved[s]}. "
        "Nếu data.yaml trỏ vào /tmp cũ đã mất, hãy set DATA_YAML_OVERRIDE tới dataset runtime V6.4 thật."
    )

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def all_images(p):
    if p is None or not Path(p).exists():
        return []
    p = Path(p)
    if p.is_file():
        raise RuntimeError(f"Split path đang là file/list, notebook cần folder ảnh: {p}")
    return sorted([x for x in p.rglob("*") if x.suffix.lower() in IMG_EXTS])

def infer_label_dir(image_dir, split):
    image_dir = Path(image_dir)
    parts = list(image_dir.parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images")
        lp = Path(*parts[:i], "labels", *parts[i+1:])
        if lp.exists():
            return lp
    return DATA_BASE / "labels" / split

def label_path_for_image(ip, image_dir, label_dir):
    rel = ip.relative_to(image_dir)
    return label_dir / rel.with_suffix(".txt")

split_info = {}
for s in ["train", "val", "test"]:
    img_dir = resolved.get(s)
    if img_dir is None or not Path(img_dir).exists():
        split_info[s] = {"img_dir": img_dir, "label_dir": None, "images": []}
        continue
    imgs = all_images(img_dir)
    lbl_dir = infer_label_dir(img_dir, s)
    split_info[s] = {"img_dir": Path(img_dir), "label_dir": lbl_dir, "images": imgs}
    print(f"{s:5s}: images={len(imgs):6d} | labels_dir={lbl_dir} | exists={lbl_dir.exists()}")

def parse_label(lp):
    rows, errors = [], []
    if not lp.exists():
        return rows, errors, "missing"
    txt = lp.read_text(errors="ignore").strip()
    if not txt:
        return rows, errors, "empty"

    for li, line in enumerate(txt.splitlines(), start=1):
        p = line.strip().split()
        if len(p) < 5:
            errors.append((li, "too_few_fields", line))
            continue
        try:
            cid = int(float(p[0]))
            xc, yc, w, h = map(float, p[1:5])
        except Exception:
            errors.append((li, "parse_error", line))
            continue

        if cid < 0 or cid >= len(NAMES):
            errors.append((li, "bad_class", line))
        if not all(np.isfinite([xc, yc, w, h])):
            errors.append((li, "non_finite", line))
        if w <= 0 or h <= 0:
            errors.append((li, "non_positive_wh", line))
        if xc < -1e-6 or xc > 1+1e-6 or yc < -1e-6 or yc > 1+1e-6:
            errors.append((li, "center_out_of_range", line))
        if w > 1+1e-6 or h > 1+1e-6:
            errors.append((li, "wh_out_of_range", line))

        x1, y1 = xc-w/2, yc-h/2
        x2, y2 = xc+w/2, yc+h/2
        overflow = max(0.0, -x1, -y1, x2-1.0, y2-1.0)

        rows.append({
            "cid": cid, "xc": xc, "yc": yc, "w": w, "h": h,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "overflow": overflow, "raw": line.strip(),
        })

    return rows, errors, "ok"

label_records, label_errors, image_status = [], [], []

for s in ["train", "val", "test"]:
    info = split_info[s]
    if not info["images"]:
        continue

    for ip in tqdm(info["images"], desc=f"label-scan {s}"):
        lp = label_path_for_image(ip, info["img_dir"], info["label_dir"])
        rows, errs, status = parse_label(lp)
        image_status.append({
            "split": s, "image": str(ip), "label": str(lp),
            "label_status": status, "n_boxes": len(rows),
        })

        for e in errs:
            label_errors.append({
                "split": s, "image": str(ip), "label": str(lp),
                "line": e[0], "error": e[1], "raw": e[2],
            })

        for j, r in enumerate(rows):
            rr = dict(r)
            rr.update({
                "split": s, "image": str(ip), "label": str(lp),
                "gt_index": j,
                "class": NAMES[r["cid"]] if 0 <= r["cid"] < len(NAMES) else str(r["cid"]),
            })
            label_records.append(rr)

df_boxes = pd.DataFrame(label_records)
df_err = pd.DataFrame(label_errors)
df_img = pd.DataFrame(image_status)

print("\nImage/label status:")
print(df_img.groupby(["split", "label_status"]).size().unstack(fill_value=0))

print("\nBoxes per split/class:")
if len(df_boxes):
    print(pd.crosstab(df_boxes["split"], df_boxes["class"]))

print("\nLabel errors:", len(df_err))
if len(df_err):
    display(df_err.head(100))

df_boxes.to_csv(OUT / "bbox_records.csv.gz", index=False, compression="gzip")
df_img.to_csv(OUT / "image_label_status.csv", index=False)
df_err.to_csv(OUT / "label_errors.csv", index=False)

In [ ]:
def box_iou_xyxy(a, b):
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    iw = max(0.0, ix2-ix1); ih = max(0.0, iy2-iy1)
    inter = iw*ih
    aa = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    bb = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    u = aa + bb - inter
    return inter/u if u > 0 else 0.0

if len(df_boxes):
    df_boxes["area"] = df_boxes["w"] * df_boxes["h"]
    df_boxes["aspect"] = df_boxes["w"] / df_boxes["h"].clip(lower=1e-12)
    df_boxes["edge_touch"] = (
        (df_boxes["x1"] <= 1e-6) |
        (df_boxes["y1"] <= 1e-6) |
        (df_boxes["x2"] >= 1-1e-6) |
        (df_boxes["y2"] >= 1-1e-6)
    )
    df_boxes["tiny_lt_0p1pct"] = df_boxes["area"] < 0.001

geom = (
    df_boxes.groupby(["split", "class"])
    .agg(
        boxes=("area", "size"),
        median_area=("area", "median"),
        p10_area=("area", lambda x: x.quantile(.10)),
        p90_area=("area", lambda x: x.quantile(.90)),
        median_aspect=("aspect", "median"),
        edge_touch_rate=("edge_touch", "mean"),
        tiny_rate=("tiny_lt_0p1pct", "mean"),
        median_overflow=("overflow", "median"),
        max_overflow=("overflow", "max"),
    )
    .reset_index()
) if len(df_boxes) else pd.DataFrame()

display(geom)
geom.to_csv(OUT / "bbox_geometry_profile.csv", index=False)

dup_rows = []
if len(df_boxes):
    for (split, image), g in tqdm(df_boxes.groupby(["split", "image"]), desc="duplicate-GT scan"):
        rows = g.to_dict("records")
        for i in range(len(rows)):
            for j in range(i+1, len(rows)):
                if rows[i]["cid"] != rows[j]["cid"]:
                    continue
                a = [rows[i]["x1"], rows[i]["y1"], rows[i]["x2"], rows[i]["y2"]]
                b = [rows[j]["x1"], rows[j]["y1"], rows[j]["x2"], rows[j]["y2"]]
                z = box_iou_xyxy(a, b)
                if z >= DUP_GT_IOU:
                    dup_rows.append({
                        "split": split, "image": image, "class": rows[i]["class"],
                        "gt_i": rows[i]["gt_index"], "gt_j": rows[j]["gt_index"], "iou": z,
                    })

df_dupgt = pd.DataFrame(dup_rows)
print("Duplicate-GT candidates:", len(df_dupgt))
if len(df_dupgt):
    display(df_dupgt.head(100))
df_dupgt.to_csv(OUT / "duplicate_gt_candidates.csv", index=False)

In [ ]:
hash_rows = []
for s in ["train", "val", "test"]:
    for ip in tqdm(split_info[s]["images"], desc=f"SHA256 {s}"):
        hash_rows.append({
            "split": s,
            "image": str(ip),
            "sha256": hashlib.sha256(ip.read_bytes()).hexdigest(),
        })

df_hash = pd.DataFrame(hash_rows)
df_hash.to_csv(OUT / "image_sha256.csv.gz", index=False, compression="gzip")

exact_cross = []
if len(df_hash):
    for sha, g in df_hash.groupby("sha256"):
        splits = sorted(set(g["split"]))
        if len(splits) > 1:
            exact_cross.append({
                "sha256": sha,
                "splits": ",".join(splits),
                "images": " | ".join(g["image"].tolist()),
            })

df_exact = pd.DataFrame(exact_cross)
print("Exact cross-split duplicate groups:", len(df_exact))
df_exact.to_csv(OUT / "cross_split_exact_duplicates.csv", index=False)

crop_marker_rows = []
for s in ["train", "val", "test"]:
    for ip in split_info[s]["images"]:
        n = ip.name.lower()
        if "v6p4crop" in n or "_crop_" in n or "hardcrop" in n:
            crop_marker_rows.append({"split": s, "image": str(ip)})

df_crop_marker = pd.DataFrame(crop_marker_rows)
if len(df_crop_marker):
    print("\nCrop-marker counts:")
    print(df_crop_marker["split"].value_counts())
else:
    print("\nCrop-marker counts: none")

bad_crop_split = (
    df_crop_marker[df_crop_marker["split"].isin(["val", "test"])]
    if len(df_crop_marker) else pd.DataFrame()
)
bad_crop_split.to_csv(OUT / "crop_derivative_outside_train.csv", index=False)

near_rows = []
if RUN_PHASH_NEAR_DUP:
    ph_rows = []
    for s in ["train", "val", "test"]:
        for ip in tqdm(split_info[s]["images"], desc=f"pHash {s}"):
            try:
                with Image.open(ip) as im:
                    ph = imagehash.phash(im.convert("RGB"), hash_size=8)
                ph_rows.append({
                    "split": s, "image": str(ip),
                    "phash_hex": str(ph), "phash_int": int(str(ph), 16),
                })
            except Exception:
                ph_rows.append({
                    "split": s, "image": str(ip),
                    "phash_hex": None, "phash_int": None,
                })

    df_ph = pd.DataFrame(ph_rows)
    df_ph.to_csv(OUT / "image_phash.csv.gz", index=False, compression="gzip")

    valid = df_ph.dropna(subset=["phash_int"]).copy()
    valid["phash_int"] = valid["phash_int"].map(int)

    # Hamming <= 4, chia 5 band thì ít nhất 1 band phải giống hệt.
    widths = [13, 13, 13, 13, 12]
    shifts = []
    cur = 0
    for w in widths:
        shifts.append(cur)
        cur += w

    buckets = {}
    for idx, r in valid.iterrows():
        v = int(r["phash_int"])
        for band, (w, sh) in enumerate(zip(widths, shifts)):
            key = (band, (v >> sh) & ((1 << w) - 1))
            buckets.setdefault(key, []).append(idx)

    candidate_pairs = set()
    for ids in buckets.values():
        if len(ids) < 2:
            continue
        for ai in range(len(ids)):
            for bi in range(ai+1, len(ids)):
                a, b = ids[ai], ids[bi]
                if valid.at[a, "split"] == valid.at[b, "split"]:
                    continue
                candidate_pairs.add((min(a,b), max(a,b)))

    for a, b in tqdm(candidate_pairs, desc="pHash candidate compare"):
        va = int(valid.at[a, "phash_int"])
        vb = int(valid.at[b, "phash_int"])
        hd = (va ^ vb).bit_count()
        if hd <= PHASH_MAX_HAMMING:
            near_rows.append({
                "split_a": valid.at[a, "split"],
                "image_a": valid.at[a, "image"],
                "split_b": valid.at[b, "split"],
                "image_b": valid.at[b, "image"],
                "phash_hamming": hd,
            })

df_near = pd.DataFrame(near_rows)
print("pHash cross-split pairs <=", PHASH_MAX_HAMMING, ":", len(df_near))
if len(df_near):
    display(df_near.head(100))
df_near.to_csv(OUT / "cross_split_phash_near_duplicates.csv", index=False)

In [ ]:
gate = {
    "data_yaml": str(DATA_YAML),
    "weights": str(WEIGHTS),
    "split_counts": {s: len(split_info[s]["images"]) for s in ["train", "val", "test"]},
    "boxes": int(len(df_boxes)),
    "label_errors": int(len(df_err)),
    "duplicate_gt_candidates_iou_ge_0p95": int(len(df_dupgt)),
    "cross_split_exact_duplicate_groups": int(len(df_exact)),
    "cross_split_phash_pairs": int(len(df_near)),
    "crop_derivatives_in_val_test": int(len(bad_crop_split)),
}

gate["hard_fail"] = bool(
    gate["label_errors"] > 0 or
    gate["cross_split_exact_duplicate_groups"] > 0 or
    gate["crop_derivatives_in_val_test"] > 0
)

gate["warnings"] = []
if gate["duplicate_gt_candidates_iou_ge_0p95"] > 0:
    gate["warnings"].append("duplicate_gt_candidates")
if gate["cross_split_phash_pairs"] > 0:
    gate["warnings"].append("near_duplicate_manual_review")
if len(df_img) and (df_img["label_status"] == "missing").sum() > 0:
    gate["warnings"].append("missing_label_files_may_be_background")

print(json.dumps(gate, indent=2))
(OUT / "dataset_gate.json").write_text(json.dumps(gate, indent=2))

In [ ]:
strict_report = {}

if RUN_MODEL_VAL:
    model = YOLO(str(WEIGHTS))

    metrics = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=IMGSZ,
        batch=VAL_BATCH,
        device=DEVICE,
        plots=False,
        verbose=False,
    )

    m = metrics.box
    all_ap = np.asarray(m.all_ap, dtype=float)
    ap50 = np.asarray(m.ap50, dtype=float)
    class_ids = [int(x) for x in np.asarray(m.ap_class_index).tolist()]

    if all_ap.ndim == 1:
        all_ap = all_ap[:, None]

    thresholds = np.arange(0.50, 0.50 + 0.05 * all_ap.shape[1], 0.05)

    def col_for(t):
        return int(np.argmin(np.abs(thresholds - t)))

    per_class = []
    for row_i, cid in enumerate(class_ids):
        vals = all_ap[row_i]
        ap5095 = float(np.mean(vals))
        d = {
            "cid": cid,
            "class": NAMES[cid],
            "AP50": float(ap50[row_i]),
            "AP50_95": ap5095,
            "gap_50_to_50_95": float(ap50[row_i] - ap5095),
        }
        for t in [0.50, 0.75, 0.90, 0.95]:
            d[f"AP{int(t*100)}"] = (
                float(vals[col_for(t)]) if all_ap.shape[1] > 1 else np.nan
            )
        per_class.append(d)

    df_strict = pd.DataFrame(per_class)
    display(df_strict)

    global_by_iou = {}
    if all_ap.shape[1] > 1:
        for j, t in enumerate(thresholds):
            global_by_iou[f"AP{int(round(t*100))}"] = float(np.mean(all_ap[:, j]))

    strict_report = {
        "mAP50": float(m.map50),
        "mAP50_95": float(m.map),
        "gap": float(m.map50 - m.map),
        "precision": float(m.mp),
        "recall": float(m.mr),
        "target_gap_le_0p30": bool((m.map50 - m.map) <= 0.30),
        "global_by_iou": global_by_iou,
        "per_class": per_class,
    }

    print("\nGLOBAL STRICT-IoU:")
    print(json.dumps(strict_report, indent=2))

    df_strict.to_csv(OUT / "strict_iou_per_class.csv", index=False)
    (OUT / "strict_iou_report.json").write_text(json.dumps(strict_report, indent=2))

In [ ]:
def gt_rows_for_label(lp):
    rows, errs, status = parse_label(lp)
    return rows

def center(box):
    return np.array([(box[0]+box[2])/2, (box[1]+box[3])/2], dtype=float)

def wh(box):
    return np.array([max(0.0, box[2]-box[0]), max(0.0, box[3]-box[1])], dtype=float)

def center_dist(a, b):
    return float(np.linalg.norm(center(a)-center(b)))

def centered_pred_box(pred_box, gt_box):
    c = center(gt_box)
    w, h = wh(pred_box)
    return np.array([c[0]-w/2, c[1]-h/2, c[0]+w/2, c[1]+h/2], dtype=float)

def area(box):
    w, h = wh(box)
    return float(w*h)

def greedy_match(gts, preds):
    # One-to-one same-class diagnostic matching.
    pairs = []
    for gi, g in enumerate(gts):
        gb = g["xyxy"]
        for pi, p in enumerate(preds):
            if g["cid"] != p["cid"]:
                continue
            pb = p["xyxy"]
            cd = center_dist(gb, pb)
            z = box_iou_xyxy(gb, pb)
            if cd > CENTER_MATCH_MAX and z < 0.05:
                continue
            score = z - 0.25*cd
            pairs.append((score, gi, pi, z, cd))

    pairs.sort(reverse=True)
    used_g, used_p, matches = set(), set(), {}

    for score, gi, pi, z, cd in pairs:
        if gi in used_g or pi in used_p:
            continue
        used_g.add(gi)
        used_p.add(pi)
        matches[gi] = (pi, z, cd)

    return matches

residual_rows = []

if RUN_RESIDUAL_ANALYSIS:
    if "model" not in globals():
        model = YOLO(str(WEIGHTS))

    val_imgs = split_info["val"]["images"]
    val_img_dir = split_info["val"]["img_dir"]
    val_lbl_dir = split_info["val"]["label_dir"]

    for start in tqdm(range(0, len(val_imgs), RESIDUAL_BATCH), desc="residual batches"):
        batch_paths = val_imgs[start:start+RESIDUAL_BATCH]

        results = model.predict(
            source=[str(p) for p in batch_paths],
            imgsz=IMGSZ,
            conf=RESIDUAL_CONF,
            device=DEVICE,
            verbose=False,
        )

        for ip, res in zip(batch_paths, results):
            lp = label_path_for_image(ip, val_img_dir, val_lbl_dir)
            gt_raw = gt_rows_for_label(lp)

            gts = []
            for j, g in enumerate(gt_raw):
                gts.append({
                    "gt_index": j,
                    "cid": int(g["cid"]),
                    "cls": NAMES[int(g["cid"])],
                    "xyxy": np.array([g["x1"], g["y1"], g["x2"], g["y2"]], dtype=float),
                })

            preds = []
            if res.boxes is not None and len(res.boxes):
                cids = res.boxes.cls.detach().cpu().numpy().astype(int)
                confs = res.boxes.conf.detach().cpu().numpy()
                boxes = res.boxes.xyxyn.detach().cpu().numpy()

                for j in range(len(cids)):
                    preds.append({
                        "pred_index": j,
                        "cid": int(cids[j]),
                        "cls": NAMES[int(cids[j])],
                        "conf": float(confs[j]),
                        "xyxy": boxes[j].astype(float),
                    })

            matches = greedy_match(gts, preds)

            for gi, g in enumerate(gts):
                gb = g["xyxy"]

                if gi not in matches:
                    residual_rows.append({
                        "image": str(ip), "label": str(lp),
                        "gt_index": gi, "cid": g["cid"], "class": g["cls"],
                        "status": "DETECTION_MISS",
                        "pred_index": None, "pred_conf": None,
                        "actual_iou": 0.0, "centered_iou": 0.0,
                        "center_dist": None, "area_ratio": None,
                        "width_ratio": None, "height_ratio": None,
                        "gt_xyxy": gb.tolist(), "pred_xyxy": None,
                    })
                    continue

                pi, z, cd = matches[gi]
                p = preds[pi]
                pb = p["xyxy"]
                cpb = centered_pred_box(pb, gb)
                cz = box_iou_xyxy(gb, cpb)

                gw, gh = wh(gb)
                pw, ph = wh(pb)

                ar = area(pb) / max(area(gb), 1e-12)
                wr = pw / max(gw, 1e-12)
                hr = ph / max(gh, 1e-12)

                if z >= GOOD_IOU:
                    status = "GOOD75"
                elif cz >= GOOD_IOU:
                    status = "CENTER_ERROR"
                else:
                    status = "EXTENT_SHAPE_ERROR"

                residual_rows.append({
                    "image": str(ip), "label": str(lp),
                    "gt_index": gi, "cid": g["cid"], "class": g["cls"],
                    "status": status,
                    "pred_index": p["pred_index"], "pred_conf": p["conf"],
                    "actual_iou": float(z), "centered_iou": float(cz),
                    "center_dist": float(cd), "area_ratio": float(ar),
                    "width_ratio": float(wr), "height_ratio": float(hr),
                    "gt_xyxy": gb.tolist(), "pred_xyxy": pb.tolist(),
                })

df_res = pd.DataFrame(residual_rows)
df_res.to_csv(OUT / "val_residual_localization.csv.gz", index=False, compression="gzip")

if len(df_res):
    residual_summary = (
        df_res.groupby(["class", "status"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    class_diag = []
    for cls, g in df_res.groupby("class"):
        matched = g[g["status"] != "DETECTION_MISS"]
        class_diag.append({
            "class": cls,
            "gt": len(g),
            "miss_rate": float((g["status"] == "DETECTION_MISS").mean()),
            "actual_pass75": float((g["actual_iou"] >= 0.75).mean()),
            "centered_pass75": float((g["centered_iou"] >= 0.75).mean()),
            "center_uplift75": float(
                (g["centered_iou"] >= 0.75).mean() -
                (g["actual_iou"] >= 0.75).mean()
            ),
            "median_actual_iou": float(g["actual_iou"].median()),
            "median_centered_iou": float(g["centered_iou"].median()),
            "median_area_ratio_matched": float(matched["area_ratio"].median()) if len(matched) else np.nan,
            "p10_area_ratio_matched": float(matched["area_ratio"].quantile(.10)) if len(matched) else np.nan,
            "p90_area_ratio_matched": float(matched["area_ratio"].quantile(.90)) if len(matched) else np.nan,
            "center_error_rate": float((g["status"] == "CENTER_ERROR").mean()),
            "extent_shape_error_rate": float((g["status"] == "EXTENT_SHAPE_ERROR").mean()),
        })

    df_diag = pd.DataFrame(class_diag).sort_values("actual_pass75")
    display(residual_summary)
    display(df_diag)

    residual_summary.to_csv(OUT / "val_residual_status_summary.csv", index=False)
    df_diag.to_csv(OUT / "val_residual_class_diagnostics.csv", index=False)
else:
    df_diag = pd.DataFrame()

In [ ]:
CONTACT_DIR = OUT / "contact_sheets"
CONTACT_DIR.mkdir(parents=True, exist_ok=True)

try:
    font = ImageFont.truetype("DejaVuSans.ttf", 17)
    small = ImageFont.truetype("DejaVuSans.ttf", 14)
except Exception:
    font = ImageFont.load_default()
    small = font

def draw_norm_box(draw, box, W, H, offset_y, outline, width=4, text=None):
    x1, y1, x2, y2 = box
    rect = [x1*W, y1*H+offset_y, x2*W, y2*H+offset_y]
    draw.rectangle(rect, outline=outline, width=width)
    if text:
        draw.text((rect[0]+3, max(offset_y, rect[1]-18)), text, fill=outline, font=small)

if len(df_res):
    hard = df_res[df_res["status"].isin([
        "DETECTION_MISS", "CENTER_ERROR", "EXTENT_SHAPE_ERROR"
    ])].copy()

    hard["_priority"] = (
        (hard["status"] == "EXTENT_SHAPE_ERROR").astype(int)*3 +
        (hard["status"] == "CENTER_ERROR").astype(int)*2 +
        hard["pred_conf"].fillna(0)
    )

    hard = hard.sort_values(
        ["_priority", "class"], ascending=[False, True]
    ).head(MAX_CONTACT_SHEETS)

    for idx, r in hard.iterrows():
        ip = Path(r["image"])
        try:
            im = Image.open(ip).convert("RGB")
        except Exception:
            continue

        max_side = 1100
        scale = min(1.0, max_side / max(im.size))
        if scale < 1:
            im = im.resize((int(im.width*scale), int(im.height*scale)))

        W, H = im.size
        head = 90

        canvas = Image.new("RGB", (W, H+head), "white")
        canvas.paste(im, (0, head))
        d = ImageDraw.Draw(canvas)

        d.text(
            (8,8),
            f"{r['class']} | {r['status']} | actual={r['actual_iou']:.3f} | centered={r['centered_iou']:.3f}",
            fill="black", font=font
        )
        d.text(
            (8,36),
            f"GT=red | PRED=green | conf={r['pred_conf']} | {ip.name}",
            fill="black", font=small
        )
        d.text(
            (8,60),
            f"area ratio={r['area_ratio']} | center dist={r['center_dist']}",
            fill="black", font=small
        )

        gt = r["gt_xyxy"]
        if isinstance(gt, str):
            gt = json.loads(gt.replace("'", '"'))
        draw_norm_box(d, gt, W, H, head, "red", 4, "GT")

        pb = r["pred_xyxy"]
        if pb is not None and not (isinstance(pb, float) and np.isnan(pb)):
            if isinstance(pb, str):
                pb = json.loads(pb.replace("'", '"'))
            draw_norm_box(d, pb, W, H, head, "green", 4, "PRED")

        name = (
            f"{r['class']}_{r['status']}_"
            f"{Path(r['image']).stem}_{int(r['gt_index'])}.jpg"
        )
        canvas.save(CONTACT_DIR / name, quality=90)

print("Contact sheets:", CONTACT_DIR)

In [ ]:
decision = {
    "dataset_gate": gate,
    "strict_iou": strict_report,
    "next_action": [],
    "notes": [],
}

if gate["hard_fail"]:
    decision["next_action"].append("STOP_AND_FIX_DATASET_INTEGRITY")
else:
    decision["notes"].append("Dataset integrity hard gate passed.")

if len(df_dupgt) > 0:
    decision["next_action"].append("MANUAL_REVIEW_DUPLICATE_GT_CANDIDATES")

if len(df_near) > 0:
    decision["next_action"].append("MANUAL_REVIEW_CROSS_SPLIT_NEAR_DUPLICATES")

if strict_report:
    if strict_report["gap"] <= 0.30:
        decision["next_action"].append("GAP_TARGET_MET_ON_VAL")
    else:
        decision["next_action"].append("STRICT_LOCALIZATION_GAP_STILL_HIGH")

if len(df_diag):
    focus_classes = [
        "insulator_broken",
        "insulator_flashover",
        "broken_strand",
        "bird_nest",
    ]
    focus = df_diag[df_diag["class"].isin(focus_classes)]

    for _, r in focus.iterrows():
        cls = r["class"]

        if r["miss_rate"] >= 0.20:
            decision["next_action"].append(f"{cls}:DETECTION_COVERAGE_OR_RECALL")

        if r["center_uplift75"] >= 0.10:
            decision["next_action"].append(f"{cls}:CENTER_LOCALIZATION_DOMINANT")

        if r["extent_shape_error_rate"] >= 0.20:
            decision["next_action"].append(f"{cls}:EXTENT_SHAPE_REVIEW")

    decision["residual_diagnostics"] = df_diag.to_dict("records")

if not gate["hard_fail"] and strict_report:
    if strict_report["gap"] > 0.30:
        decision["notes"].append(
            "Nếu contact sheets cho thấy GT V6.4 hợp lý mà residual vẫn lặp lại, "
            "không tiếp tục scale bbox. Khi đó chuyển sang model/localization side."
        )
        decision["notes"].append(
            "Nếu EXTENT_SHAPE_ERROR tập trung vào một annotation convention, "
            "review policy class đó trước khi train architecture mới."
        )
    else:
        decision["notes"].append(
            "Gap <= 0.30 trên VAL: giữ V6.4 làm canonical candidate."
        )

seen = set()
dedup = []
for x in decision["next_action"]:
    if x not in seen:
        seen.add(x)
        dedup.append(x)
decision["next_action"] = dedup

report_path = OUT / "v6p4_post_clean_decision.json"
report_path.write_text(json.dumps(decision, indent=2, default=str))

print("="*90)
print("FINAL DECISION")
print("="*90)
print(json.dumps(decision, indent=2, default=str))

print("\nSaved:", report_path)
print("\nArtifacts:")
for p in sorted(OUT.iterdir()):
    print(" -", p)

## Đọc kết quả thế nào

Quan trọng nhất:

- `dataset_gate.json`: lỗi integrity/leakage cơ bản.
- `strict_iou_report.json`: mAP50, mAP50:95, AP75/AP90/AP95 và gap trên VAL.
- `strict_iou_per_class.csv`: class nào kéo strict localization xuống.
- `val_residual_class_diagnostics.csv`: detection miss vs center error vs extent/shape error.
- `contact_sheets/`: GT đỏ, prediction xanh.
- `v6p4_post_clean_decision.json`: kết luận diagnostic.

`EXTENT_SHAPE_ERROR` **không tự động có nghĩa là GT sai**. Nó chỉ nói rằng kể cả đưa prediction về đúng tâm GT thì IoU vẫn thấp. Phải nhìn contact sheet để phân biệt model sai với annotation extent chưa ổn.